# Airline Passenger Forecasting using a Seasonally Adjusted LSTM

This notebook documents the portfolio workflow used by the production code under `src/`. It uses chronological splitting, training-only scaling, year-over-year log-growth features, a compact Keras LSTM, baseline comparison, and recursive future forecasting.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path
import sys
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.data_preprocessing import load_and_prepare
from src.feature_engineering import add_eda_features
from src.sequence_generation import fit_growth_scaler, build_sequence_dataset, chronological_masks
from src.model_training import build_lstm_model, train_model, set_reproducible_seed
from src.model_evaluation import predict_sequence_subset, calculate_metrics, baseline_predictions, comparison_table
from src.forecasting_pipeline import recursive_forecast, summarize_forecast

print("Project root:", PROJECT_ROOT)

## 1. Load and validate the monthly series

In [ ]:
frame, preprocessing_notes = load_and_prepare(SAMPLE_DATA_PATH)
print(frame.shape)
print(preprocessing_notes)
frame.head()

## 2. Trend and seasonality analysis

In [ ]:
eda = add_eda_features(frame)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eda["Month"], eda["Passengers"])
ax.set(title="Monthly Airline Passenger Demand", xlabel="Month", ylabel="Passengers (thousands)")
ax.grid(alpha=0.25)
plt.show()

seasonal = eda.groupby("MonthNumber")["Passengers"].mean()
seasonal.plot(marker="o", figsize=(10, 4), title="Average Seasonal Pattern")
plt.ylabel("Average passengers (thousands)")
plt.grid(alpha=0.25)
plt.show()

## 3. Leakage-safe split and feature construction

The scaler is fitted on training seasonal-growth values only. Targets are split chronologically into training, validation, and test periods.

In [ ]:
train_end = 96
validation_end = 120
scaler = fit_growth_scaler(frame, train_end=train_end, seasonal_period=12)
dataset = build_sequence_dataset(frame, scaler, lookback=12, seasonal_period=12)
train_mask, validation_mask, test_mask = chronological_masks(dataset.target_indices, train_end, validation_end)

print("X shape:", dataset.X.shape)
print("Training sequences:", train_mask.sum())
print("Validation sequences:", validation_mask.sum())
print("Test sequences:", test_mask.sum())

## 4. LSTM architecture

In [ ]:
model = build_lstm_model(lookback=12, n_features=3, lstm_units=16, dropout=0.10, dense_units=8, learning_rate=0.003)
model.summary()

## 5. Train or load the packaged model

Set `RUN_TRAINING = True` to regenerate the model. The default loads the committed inference artifact.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    set_reproducible_seed(42)
    history = train_model(
        model, dataset.X[train_mask], dataset.y[train_mask],
        dataset.X[validation_mask], dataset.y[validation_mask],
        epochs=150, batch_size=8, patience=12, verbose=1
    )
    model.save(MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
else:
    import keras
    model = keras.saving.load_model(MODEL_PATH, compile=False)
    scaler = joblib.load(SCALER_PATH)

print("Model ready.")

## 6. Final test evaluation

In [ ]:
test_predictions = predict_sequence_subset(model, dataset, test_mask, frame, scaler, seasonal_period=12)
metrics = calculate_metrics(test_predictions["Actual"], test_predictions["Predicted"])
metrics.to_dict()

In [ ]:
test_predictions.set_index("Month")[["Actual", "Predicted"]].plot(figsize=(12, 5), title="Actual vs Predicted — Final Test Period")
plt.ylabel("Passengers (thousands)")
plt.grid(alpha=0.25)
plt.show()

## 7. Baseline comparison

In [ ]:
test_indices = dataset.target_indices[test_mask]
baselines = baseline_predictions(frame, test_indices, fit_end=validation_end, seasonal_period=12)
comparison = comparison_table(test_predictions["Actual"].to_numpy(), test_predictions["Predicted"].to_numpy(), baselines)
comparison

## 8. Recursive future forecast

In [ ]:
forecast = recursive_forecast(frame, model, scaler, horizon=24, lookback=12, seasonal_period=12)
summary = summarize_forecast(forecast)
print(summary)
forecast.head()

In [ ]:
ax = frame.tail(60).set_index("Month")["Passengers"].plot(figsize=(12, 5), label="Historical")
forecast.set_index("Month")["Forecasted_Passengers"].plot(ax=ax, style="--", label="Forecast")
ax.set(title="Historical Demand and 24-Month Forecast", ylabel="Passengers (thousands)")
ax.grid(alpha=0.25)
plt.legend()
plt.show()

## 9. Portfolio conclusion

The final solution demonstrates leakage-aware forecasting, annual-seasonality handling, compact LSTM design, baseline benchmarking, saved inference artifacts, and deployment through Streamlit. For production use, add external demand drivers, prediction intervals, walk-forward validation, and monitoring.